# 11 — Strategy Lab: a head-to-head arena with honest findings

**Phase 1 portfolio artifact.** This notebook is a self-contained *research memo*:
it loads the demo lake, runs three systematic strategies across three transaction-cost
regimes against an equal-weight buy-and-hold benchmark, and records evidence-backed
**FindingCards** — **including negative results**.

> A defensible negative is a stronger portfolio line than a cherry-picked Sharpe.

**Status:** the lake holds *synthetic* data by default (`make demo`), so the verdicts
below are intentionally sobering — a geometric random walk carries no exploitable
signal by construction. Re-run `equity demo seed --real` for market data and every
card regenerates identically. Leakage-free purged walk-forward CV and an
XGBoost/LightGBM model comparison land in **Phase 2**.

## §0 Setup

Run with the backtesting engine group (and `viz` for plots):

```bash
uv sync --group backtesting --group viz
uv run jupyter lab notebooks/11-strategy-lab.ipynb
```

Plots degrade gracefully to a printed note if the `viz` group is absent.

In [1]:
import sys
from datetime import date
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import duckdb
import polars as pl

from equity_lake.core.logging import setup_structured_logging
setup_structured_logging(level="ERROR")  # keep the memo readable; surface only real errors

from equity_lake.backtesting.arena import run_arena
from equity_lake.backtesting.report import drawdown_series, write_arena_artifacts
from equity_lake.core.config import TickerConfig
from equity_lake.core.paths import DATA_DIR
from equity_lake.backtesting.evidence_view import finding_cards as load_finding_cards
from equity_lake.storage.lake_reader import duckdb_scan_for

pl.Config(fmt_str_lengths=50, tbl_cols=8)

# `make demo` seeds the auxiliary sample lake (ADR-0020), not data/lake.
DEMO_LAKE = DATA_DIR / "sample"
DEMO_BRONZE = DEMO_LAKE / "01_bronze" / "market_data" / "us_equity"

## §1 Hypothesis

Three falsifiable claims, each becoming a `FindingCard` with a `positive` /
`negative` / `inconclusive` verdict:

1. **Strategy** — at least one systematic rule (momentum / mean-reversion / trend)
   beats an equal-weight buy-and-hold benchmark on risk-adjusted return.
2. **Cost** — realistic transaction costs (≈14.3 bps fee + 30 bps tax) materially
   degrade Sharpe versus a zero-cost backtest.
3. **Benchmark** — no passive equal-weight benchmark should lose to a tuned rule
   after costs.

In [2]:
HYPOTHESES = {
    "strategy": "a systematic rule beats equal-weight buy-and-hold on Sharpe",
    "cost": "realistic costs materially degrade Sharpe vs zero-cost",
    "benchmark": "no equal-weight benchmark should lose to a rule after costs",
}
HYPOTHESES

{'strategy': 'a systematic rule beats equal-weight buy-and-hold on Sharpe',
 'cost': 'realistic costs materially degrade Sharpe vs zero-cost',
 'benchmark': 'no equal-weight benchmark should lose to a rule after costs'}

## §2 Data

Universe = the first eight `demo`-group tickers (kept small for readable equity
curves; the arena generalises to the full 50). Coverage is read straight off the
Delta-partitioned bronze lake with DuckDB.

In [3]:
universe = TickerConfig().get_tickers_by_group("demo")[:8]
START, END = date(2022, 1, 3), date(2026, 8, 4)
print("universe:", universe)

con = duckdb.connect()
con.execute("INSTALL delta; LOAD delta;")
scan = duckdb_scan_for(DEMO_BRONZE)
con.execute(
    f"SELECT min(date) AS first_day, max(date) AS last_day, "
    f"count(distinct ticker) AS tickers, count(*) AS rows FROM {scan}"
).fetchdf()

universe: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'BRK-B']


,first_day,last_day,tickers,rows
0,2021-11-01,2026-10-30,50,62750


## §3 Method

`run_arena()` loads the window once and runs every **strategy × cost-regime**
combination on the *same* data, then attaches an equal-weight buy-and-hold
benchmark. One failing run is logged and skipped rather than aborting the arena.

- **Strategies** — `momentum` (cross-sectional), `mean_reversion` (Bollinger),
  `trend_following` (SMA 50/200 crossover).
- **Cost regimes** — `zero`, `realistic` (14.3 bps fee + 30 bps tax), `high` (50 bps).
- **Benchmark** — equal-weight buy-and-hold of the same universe.

**Honesty note:** this is a full-window pseudo-OOS. Signal construction uses only
information available up to each bar (no lookahead), but there is no purged
embargoed walk-forward split yet — that ML rigor is Phase 2.

In [4]:
outcome = run_arena(universe, START, END, initial_cash=100_000.0, lake_dir=DEMO_LAKE)
strategies = sorted({r.strategy for r in outcome.runs})
regimes = sorted({r.cost_regime for r in outcome.runs})
print(f"{len(outcome.runs)} runs: strategies={strategies} regimes={regimes}")

{"table": "02_silver/corporate_actions/us_equity", "path": "./data/sample/02_silver/corporate_actions/us_equity", "event": "delta_read_failed", "logger": "equity_lake.storage.delta", "level": "error", "correlation_id": "1dc27686", "timestamp": "2026-10-31T05:31:48.788868Z", "exception": "Traceback (most recent call last):\n  File \"./src/equity_lake/storage/delta.py\", line 364, in read_delta\n    dt = DeltaTable(str(table_path), version=version) if version is not None else DeltaTable(str(table_path))\n                                                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"./.venv/lib/python3.12/site-packages/deltalake/table.py\", line 197, in __init__\n    self._table = RawDeltaTable(\n                  ^^^^^^^^^^^^^^\n_internal.TableNotFoundError: Invalid table location: ./data/sample/02_silver/corporate_actions/us_equity\nError: Os { code: 2, kind: NotFound, message: \"No such file or directory\" }"}
Traceback (most recent call last):


{"table": "02_silver/corporate_actions/us_equity", "path": "./data/sample/02_silver/corporate_actions/us_equity", "event": "delta_read_failed", "logger": "equity_lake.storage.delta", "level": "error", "correlation_id": "1dc27686", "timestamp": "2026-10-31T05:31:48.800764Z", "exception": "Traceback (most recent call last):\n  File \"./src/equity_lake/storage/delta.py\", line 364, in read_delta\n    dt = DeltaTable(str(table_path), version=version) if version is not None else DeltaTable(str(table_path))\n                                                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"./.venv/lib/python3.12/site-packages/deltalake/table.py\", line 197, in __init__\n    self._table = RawDeltaTable(\n                  ^^^^^^^^^^^^^^\n_internal.TableNotFoundError: Invalid table location: ./data/sample/02_silver/corporate_actions/us_equity\nError: Os { code: 2, kind: NotFound, message: \"No such file or directory\" }"}
Traceback (most recent call last):


{"table": "02_silver/corporate_actions/us_equity", "path": "./data/sample/02_silver/corporate_actions/us_equity", "event": "delta_read_failed", "logger": "equity_lake.storage.delta", "level": "error", "correlation_id": "1dc27686", "timestamp": "2026-10-31T05:31:48.808538Z", "exception": "Traceback (most recent call last):\n  File \"./src/equity_lake/storage/delta.py\", line 364, in read_delta\n    dt = DeltaTable(str(table_path), version=version) if version is not None else DeltaTable(str(table_path))\n                                                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"./.venv/lib/python3.12/site-packages/deltalake/table.py\", line 197, in __init__\n    self._table = RawDeltaTable(\n                  ^^^^^^^^^^^^^^\n_internal.TableNotFoundError: Invalid table location: ./data/sample/02_silver/corporate_actions/us_equity\nError: Os { code: 2, kind: NotFound, message: \"No such file or directory\" }"}
Traceback (most recent call last):


{"table": "02_silver/corporate_actions/us_equity", "path": "./data/sample/02_silver/corporate_actions/us_equity", "event": "delta_read_failed", "logger": "equity_lake.storage.delta", "level": "error", "correlation_id": "1dc27686", "timestamp": "2026-10-31T05:31:48.813638Z", "exception": "Traceback (most recent call last):\n  File \"./src/equity_lake/storage/delta.py\", line 364, in read_delta\n    dt = DeltaTable(str(table_path), version=version) if version is not None else DeltaTable(str(table_path))\n                                                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"./.venv/lib/python3.12/site-packages/deltalake/table.py\", line 197, in __init__\n    self._table = RawDeltaTable(\n                  ^^^^^^^^^^^^^^\n_internal.TableNotFoundError: Invalid table location: ./data/sample/02_silver/corporate_actions/us_equity\nError: Os { code: 2, kind: NotFound, message: \"No such file or directory\" }"}
Traceback (most recent call last):


{"table": "02_silver/corporate_actions/us_equity", "path": "./data/sample/02_silver/corporate_actions/us_equity", "event": "delta_read_failed", "logger": "equity_lake.storage.delta", "level": "error", "correlation_id": "1dc27686", "timestamp": "2026-10-31T05:31:48.819181Z", "exception": "Traceback (most recent call last):\n  File \"./src/equity_lake/storage/delta.py\", line 364, in read_delta\n    dt = DeltaTable(str(table_path), version=version) if version is not None else DeltaTable(str(table_path))\n                                                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"./.venv/lib/python3.12/site-packages/deltalake/table.py\", line 197, in __init__\n    self._table = RawDeltaTable(\n                  ^^^^^^^^^^^^^^\n_internal.TableNotFoundError: Invalid table location: ./data/sample/02_silver/corporate_actions/us_equity\nError: Os { code: 2, kind: NotFound, message: \"No such file or directory\" }"}
Traceback (most recent call last):


{"table": "02_silver/corporate_actions/us_equity", "path": "./data/sample/02_silver/corporate_actions/us_equity", "event": "delta_read_failed", "logger": "equity_lake.storage.delta", "level": "error", "correlation_id": "1dc27686", "timestamp": "2026-10-31T05:31:48.824274Z", "exception": "Traceback (most recent call last):\n  File \"./src/equity_lake/storage/delta.py\", line 364, in read_delta\n    dt = DeltaTable(str(table_path), version=version) if version is not None else DeltaTable(str(table_path))\n                                                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"./.venv/lib/python3.12/site-packages/deltalake/table.py\", line 197, in __init__\n    self._table = RawDeltaTable(\n                  ^^^^^^^^^^^^^^\n_internal.TableNotFoundError: Invalid table location: ./data/sample/02_silver/corporate_actions/us_equity\nError: Os { code: 2, kind: NotFound, message: \"No such file or directory\" }"}
Traceback (most recent call last):


{"table": "02_silver/corporate_actions/us_equity", "path": "./data/sample/02_silver/corporate_actions/us_equity", "event": "delta_read_failed", "logger": "equity_lake.storage.delta", "level": "error", "correlation_id": "1dc27686", "timestamp": "2026-10-31T05:31:48.829371Z", "exception": "Traceback (most recent call last):\n  File \"./src/equity_lake/storage/delta.py\", line 364, in read_delta\n    dt = DeltaTable(str(table_path), version=version) if version is not None else DeltaTable(str(table_path))\n                                                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"./.venv/lib/python3.12/site-packages/deltalake/table.py\", line 197, in __init__\n    self._table = RawDeltaTable(\n                  ^^^^^^^^^^^^^^\n_internal.TableNotFoundError: Invalid table location: ./data/sample/02_silver/corporate_actions/us_equity\nError: Os { code: 2, kind: NotFound, message: \"No such file or directory\" }"}
Traceback (most recent call last):


{"table": "02_silver/corporate_actions/us_equity", "path": "./data/sample/02_silver/corporate_actions/us_equity", "event": "delta_read_failed", "logger": "equity_lake.storage.delta", "level": "error", "correlation_id": "1dc27686", "timestamp": "2026-10-31T05:31:48.834508Z", "exception": "Traceback (most recent call last):\n  File \"./src/equity_lake/storage/delta.py\", line 364, in read_delta\n    dt = DeltaTable(str(table_path), version=version) if version is not None else DeltaTable(str(table_path))\n                                                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"./.venv/lib/python3.12/site-packages/deltalake/table.py\", line 197, in __init__\n    self._table = RawDeltaTable(\n                  ^^^^^^^^^^^^^^\n_internal.TableNotFoundError: Invalid table location: ./data/sample/02_silver/corporate_actions/us_equity\nError: Os { code: 2, kind: NotFound, message: \"No such file or directory\" }"}
Traceback (most recent call last):


9 runs: strategies=['mean_reversion', 'momentum', 'trend_following'] regimes=['high', 'realistic', 'zero']


## §4 Results

Risk-adjusted ranking across the 3×3 grid:

In [5]:
rows = []
for run in outcome.runs:
    res = run.result
    rows.append({
        "strategy": run.strategy,
        "cost_regime": run.cost_regime,
        "total_return": round(res.total_return, 4),
        "sharpe": round(res.sharpe_ratio, 3),
        "max_drawdown": round(res.max_drawdown, 4),
    })
pl.DataFrame(rows).sort("sharpe", descending=True)

strategy,cost_regime,total_return,sharpe,max_drawdown
str,str,f64,f64,f64
"""trend_following""","""zero""",0.8314,1.264,-0.1437
"""trend_following""","""realistic""",0.7506,1.162,-0.1518
"""trend_following""","""high""",0.4922,0.799,-0.1802
"""mean_reversion""","""zero""",0.5987,0.789,-0.1714
"""mean_reversion""","""realistic""",0.4237,0.579,-0.1894
"""momentum""","""zero""",0.0,0.0,0.0
"""momentum""","""realistic""",0.0,0.0,0.0
"""momentum""","""high""",0.0,0.0,0.0
"""mean_reversion""","""high""",-0.0598,-0.167,-0.2622


Equity curves under the **realistic** cost regime:

In [6]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(9, 4))
    for run in [r for r in outcome.runs if r.cost_regime == "realistic"]:
        eq = run.result.equity_curve
        ax.plot(range(len(eq)), eq.to_numpy(), label=run.strategy)
    ax.axhline(100_000, color="grey", lw=0.8, ls="--")
    ax.set_title("Equity curves — realistic cost regime")
    ax.set_xlabel("bar"); ax.set_ylabel("equity ($)"); ax.legend()
    plt.show()
except ImportError:
    print("[plots skipped] install the viz group: uv sync --group viz")

[plots skipped] install the viz group: uv sync --group viz


Drawdown of the worst-affected run (the most informative view):

In [7]:
worst_dd = min(outcome.runs, key=lambda r: r.result.max_drawdown)
dd = drawdown_series(worst_dd.result.equity_curve)
dd_min = dd.min()
print(f"{worst_dd.strategy}/{worst_dd.cost_regime}: min drawdown = {round(float(dd_min), 4) if dd_min is not None else float('nan')}")
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(9, 3))
    ax.fill_between(range(len(dd)), dd.to_numpy(), 0, color="crimson", alpha=0.4)
    ax.set_title(f"Drawdown — {worst_dd.strategy}/{worst_dd.cost_regime}")
    ax.set_xlabel("bar"); ax.set_ylabel("drawdown")
    plt.show()
except ImportError:
    print("[plots skipped] install the viz group: uv sync --group viz")

mean_reversion/high: min drawdown = -0.2622
[plots skipped] install the viz group: uv sync --group viz


## §5 Findings

`write_arena_artifacts()` finalises one immutable dated run per completed pair and
publishes the three Phase 1 `FindingCards` **inside that invocation's attempt
directory** (`data/findings/attempts/<UTC>-<attempt-id>/`), not as flat cards at the
findings root. We reload them through the read projection
(`evidence_view.finding_cards`, the same resolver behind `GET /findings`) to prove
the round-trip: an attempt-bound card wins by id and the frozen legacy flat cards
stay readable. These JSON cards are the frozen contract Phase 2/3 (and the React
surface) consume verbatim.

In [8]:
cards = write_arena_artifacts(outcome)
by_id = {c.id: c for c in load_finding_cards()}  # attempt-bound cards win by id
for c in cards:
    reloaded = by_id.get(c.id)
    assert reloaded is not None, f"{c.id} was not readable from the attempt directory"
    # write_arena_artifacts returns the cards it actually persisted, so full model
    # equality is the round-trip claim: matching verdict/metrics alone would let a
    # stale or wrong card with the same numbers pass.
    assert reloaded == c
    print(f"[{c.axis:9s}] {c.id:18s} {c.verdict:13s} — {c.conclusion}")
    print(f"            metrics: {c.metrics}")

[strategy ] strategy-comparison negative      — No strategy clears the benchmark by >0.1 Sharpe; best is trend_following at 1.16 vs 1.23.
            metrics: {'momentum.sharpe': 0.0, 'mean_reversion.sharpe': 0.578988012898981, 'trend_following.sharpe': 1.161880002159509, 'benchmark.sharpe': 1.2345638051765127}
[cost     ] cost-regime        positive      — Costs barely move momentum Sharpe (0.00 -> 0.00); turnover is low.
            metrics: {'zero.sharpe': 0.0, 'realistic.sharpe': 0.0, 'high.sharpe': 0.0}
[benchmark] vs-benchmark       negative      — No strategy beats the equal-weight benchmark after realistic costs.
            metrics: {'benchmark.sharpe': 1.2345638051765127, 'benchmark.total_return': 0.7393006694917679, 'momentum.sharpe': 0.0, 'momentum.total_return': 0.0, 'mean_reversion.sharpe': 0.578988012898981, 'mean_reversion.total_return': 0.42369231714092925, 'trend_following.sharpe': 1.161880002159509, 'trend_following.total_return': 0.7505526784990573}


## §6 Caveats & next steps

- **Synthetic data.** `make demo` seeds a geometric random walk; the negative
  verdicts are the *expected* honest outcome. Re-run `equity demo seed --real`
  then re-execute §3–§5 for market-data findings.
- **Costs are point-in-time** (fee + tax); there is no slippage, borrow, or
  market-impact model.
- **Momentum traded nothing** on this sample (flat equity) — itself a finding:
  the cross-sectional signal was never strong enough to cross its threshold.
- **Next (Phase 2):** leakage-free purged walk-forward CV, an XGBoost vs LightGBM
  comparison, and an enrichment ablation — each producing its own `FindingCard`.

---
*Reproduce from the CLI:* `make demo && equity arena run --start-date 2022-01-03 --end-date 2026-08-04`